In [ ]:
import sys

# repo root, relative to this notebook (utils/train_val_misalignment/)
sys.path.append("../..")

from utils.json_utils import extract_json
from pathlib import Path
from collections import Counter
from tqdm import tqdm

In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd


def analyse_train_val_gap(experiment_dirs):
    """
    Analyse train-validation ASR differences across experiments.

    Parameters
    ----------
    experiment_dirs : list[str]
        List of experiment root directories.
        Each directory may contain run_1, run_2, run_3, etc.

    Returns
    -------
    summary_df : pd.DataFrame
        Summary statistics for each experiment.

    details_df : pd.DataFrame
        Trial-level train/validation differences.
    """

    all_details = []
    summaries = []

    for experiment_dir in experiment_dirs:
        experiment_dir = Path(experiment_dir)

        # Find asr_over_trials.json files inside runs
        json_files = sorted(
            experiment_dir.glob("run_*/plots/plot_data/asr_over_trials.json")
        )

        if not json_files:
            print(f"Warning: no asr_over_trials.json found in {experiment_dir}")
            continue

        experiment_rows = []

        for json_path in json_files:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            train_trials = data["train"]["trials"]
            train_asr = data["train"]["asr"]

            val_trials = data["val"]["trials"]
            val_asr = data["val"]["asr"]

            train = dict(zip(train_trials, train_asr))
            val = dict(zip(val_trials, val_asr))

            common_trials = sorted(set(train) & set(val))

            run_name = json_path.parents[2].name

            for trial in common_trials:
                train_value = train[trial]
                val_value = val[trial]

                # Difference in percentage points
                diff_pp = (val_value - train_value) * 100
                abs_diff_pp = abs(diff_pp)

                row = {
                    "experiment": experiment_dir.name,
                    "run": run_name,
                    "trial": trial,
                    "train_asr": train_value,
                    "val_asr": val_value,
                    "diff_pp": diff_pp,
                    "abs_diff_pp": abs_diff_pp,
                }

                experiment_rows.append(row)
                all_details.append(row)

        # Statistics across ALL trials and runs of this experiment
        diffs = np.array([
            row["diff_pp"]
            for row in experiment_rows
        ])

        abs_diffs = np.array([
            row["abs_diff_pp"]
            for row in experiment_rows
        ])

        summaries.append({
            "experiment": experiment_dir.name,
            "n": len(experiment_rows),

            # Directional difference
            "mean_diff_pp": diffs.mean(),

            # Magnitude of disagreement
            "mean_abs_diff_pp": abs_diffs.mean(),

            # Potentially useful extras
            "median_abs_diff_pp": np.median(abs_diffs),
            "std_diff_pp": diffs.std(ddof=1),
            "max_abs_diff_pp": abs_diffs.max(),
        })

    summary_df = pd.DataFrame(summaries)
    details_df = pd.DataFrame(all_details)

    return summary_df, details_df

In [ ]:
exp_dirs = ["../../logs_final/dynamic/harmbench_split/adaptive_no_guards/gemini_3_5_flash_",
            "../../logs_final/dynamic/harmbench_split/adaptive_no_guards/kimi_k3_",
            "../../logs_final/dynamic/harmbench_split/adaptive_no_guards/llama_4_maverick_",
            "../../logs_final/dynamic/harmbench_split/adaptive_no_guards/qwen3_5_397b_a17b_"]

In [9]:
summary, details = analyse_train_val_gap(exp_dirs)

In [10]:
summary

,experiment,n,mean_diff_pp,mean_abs_diff_pp,median_abs_diff_pp,std_diff_pp,max_abs_diff_pp
0,gemini_3_5_flash_,60,-4.800000,8.533333,8.0,9.525701,24.0
1,kimi_k3_,60,-14.633333,14.966667,14.0,8.825938,38.0
2,llama_4_maverick_,60,-6.600000,8.933333,8.0,9.519294,30.0
3,qwen3_5_397b_a17b_,60,-6.433333,7.433333,6.0,7.915586,28.0


In [ ]:
exp_dirs = ["../../logs_final/dynamic/harmbench_split/adaptive_with_guards/gemini_3_5_flash_",
            "../../logs_final/dynamic/harmbench_split/adaptive_with_guards/kimi_k3_",
            "../../logs_final/dynamic/harmbench_split/adaptive_with_guards/llama_4_maverick_",
            "../../logs_final/dynamic/harmbench_split/adaptive_with_guards/qwen3_5_397b_a17b_"]

summary, details = analyse_train_val_gap(exp_dirs)
summary

## Data

In [13]:
import re
from collections import Counter
import pandas as pd


def count_categories(file_path):
    """
    Parse attack samples from a text file and count occurrences
    of each value in the "category" field.

    Returns
    -------
    pd.DataFrame
        Columns:
        - category
        - count
        - percentage
    """

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Extract all category fields
    categories = re.findall(
        r'"category"\s*:\s*"([^"]+)"',
        text
    )

    counts = Counter(categories)
    total = len(categories)

    result = pd.DataFrame(
        [
            {
                "category": category,
                "count": count,
                "percentage": count / total * 100
            }
            for category, count in counts.items()
        ]
    )

    result = result.sort_values(
        "count",
        ascending=False
    ).reset_index(drop=True)

    return result

Train

In [ ]:
path = "../../logs_final/dynamic/harmbench_split/adaptive_no_guards/gemini_3_5_flash_/run_1/attack_trial_0/trial_summary.txt"

In [15]:
count_categories(path)

,category,count,percentage
0,illegal,14,28.0
1,cybercrime_intrusion,11,22.0
2,misinformation_disinformation,8,16.0
3,harmful,7,14.0
4,harassment_bullying,5,10.0
5,chemical_biological,5,10.0
